# Lab 6 — Forecasting and Best Linear Prediction

**Course:** MATH 7339 — Machine Learning and Statistical Learning Theory 2  
**Book:** *Modern Time Series Analysis and Sequence Learning*  
**Chapter:** 6 — Forecasting Theory and Best Linear Prediction

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-06-forecasting-best-linear-prediction-lab.ipynb)

## Independent-study goal

This lab is designed so that you can study on your own. Before each programming task, you will see the mathematical idea, the meaning of the symbols, and what you should expect to observe in the output.

By the end of this lab, you should be able to:

1. Explain why forecasting is a conditional prediction problem.
2. Distinguish the best mean-square predictor from the best linear predictor.
3. Solve the normal equations for one-step-ahead linear prediction.
4. Compute theoretical forecast weights for AR(1) and MA(1) processes.
5. Build empirical lag-based linear forecasters from data.
6. Check the orthogonality principle numerically.
7. Compare a model-based forecast with a naive baseline.

We will avoid advanced packages as much as possible. The main tools are `numpy`, `pandas`, and `matplotlib`.

## 0. Mathematical background before programming

Suppose we observe a time series

$$
x_1, x_2, \ldots, x_n.
$$

The forecasting problem asks for a prediction of a future value such as $x_{n+1}$ or $x_{n+h}$.

In probability notation, the ideal one-step-ahead forecast is often written as

$$
\hat X_{n+1} = E[X_{n+1} \mid X_1, \ldots, X_n].
$$

This conditional expectation minimizes mean squared prediction error among all possible prediction functions of the observed data.

However, conditional expectations can be hard to compute. In classical time series, we often restrict attention to **linear predictors** of the form

$$
\hat X_{n+1} = a_1 X_n + a_2 X_{n-1} + \cdots + a_p X_{n-p+1}.
$$

The best linear predictor is the linear forecast with the smallest mean squared error.

The key principle is the **orthogonality principle**:

$$
E[(X_{n+1} - \hat X_{n+1})X_{n+1-k}] = 0,
\quad k=1,\ldots,p.
$$

In words: the forecast error should be uncorrelated with every variable used in the forecast.

## 1. Setup

Run the next cell first. It imports the libraries, sets a random seed, and defines a few helper functions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

def mse(y_true, y_pred):
    """Mean squared error."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean((y_true - y_pred) ** 2)


def autocovariance_from_formula_ar1(phi, sigma2, max_lag):
    """Autocovariance gamma(h) for a stationary AR(1)."""
    gamma0 = sigma2 / (1 - phi**2)
    return np.array([gamma0 * phi**abs(h) for h in range(max_lag + 1)])


def toeplitz_from_autocov(gamma, p):
    """Build the p by p Toeplitz covariance matrix using gamma(0),...,gamma(p-1)."""
    G = np.empty((p, p))
    for i in range(p):
        for j in range(p):
            G[i, j] = gamma[abs(i - j)]
    return G


def one_step_blp_weights(gamma, p):
    """
    Solve Gamma_p a = gamma_p for the best linear predictor
    of X_{t+1} using [X_t, X_{t-1}, ..., X_{t-p+1}].
    """
    Gamma = toeplitz_from_autocov(gamma, p)
    gamma_vec = np.array([gamma[k] for k in range(1, p + 1)])
    weights = np.linalg.solve(Gamma, gamma_vec)
    pred_mse = gamma[0] - gamma_vec @ weights
    return weights, pred_mse

print("Setup complete.")

## 2. Simulate a process with real forecasting structure

We start with a stationary AR(1) process:

$$
X_t = \phi X_{t-1} + W_t,
\qquad W_t \sim WN(0,\sigma^2),
\qquad |\phi| < 1.
$$

For an AR(1) process, the best one-step forecast using the past is especially simple:

$$
\hat X_{t+1} = \phi X_t.
$$

The $h$-step forecast is

$$
\hat X_{t+h} = \phi^h X_t.
$$

We will simulate an AR(1) series and use it throughout the lab.

In [ ]:
def simulate_ar1(phi=0.7, sigma=1.0, n=300, burnin=200, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    total = n + burnin
    w = rng.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(1, total):
        x[t] = phi * x[t - 1] + w[t]
    return x[burnin:]

phi_true = 0.75
sigma_true = 1.0
n = 300
x = simulate_ar1(phi=phi_true, sigma=sigma_true, n=n, rng=rng)

time = np.arange(n)
plt.plot(time, x)
plt.axhline(0, linewidth=1)
plt.title(f"Simulated stationary AR(1), phi = {phi_true}")
plt.xlabel("time")
plt.ylabel("value")
plt.show()

pd.Series(x, name="x").describe()

### Interpretation checkpoint

The process is random, so it does not repeat exactly. But because $|\phi|<1$, it tends to fluctuate around a stable mean. Positive $\phi$ creates persistence: large positive values tend to be followed by positive values, and large negative values tend to be followed by negative values.

## 3. Theoretical best linear prediction for AR(1)

For a zero-mean stationary series, the one-step linear prediction problem using $p$ lags is

$$
\hat X_{t+1} = a_1 X_t + a_2 X_{t-1} + \cdots + a_p X_{t-p+1}.
$$

The best weights solve

$$
\Gamma_p a = \gamma_p,
$$

where

$$
\Gamma_p =
\begin{pmatrix}
\gamma(0) & \gamma(1) & \cdots & \gamma(p-1) \\
\gamma(1) & \gamma(0) & \cdots & \gamma(p-2) \\
\vdots & \vdots & \ddots & \vdots \\
\gamma(p-1) & \gamma(p-2) & \cdots & \gamma(0)
\end{pmatrix},
\qquad
\gamma_p =
\begin{pmatrix}
\gamma(1) \\
\gamma(2) \\
\vdots \\
\gamma(p)
\end{pmatrix}.
$$

For an AR(1), we expect the solution to be approximately

$$
a_1 = \phi, \qquad a_2 = \cdots = a_p = 0.
$$

This means only the most recent observation is needed when the true process is AR(1).

In [ ]:
max_lag = 10
p = 5
sigma2_true = sigma_true**2

gamma_ar1 = autocovariance_from_formula_ar1(phi_true, sigma2_true, max_lag)
weights_ar1, pred_mse_ar1 = one_step_blp_weights(gamma_ar1, p)

weights_table = pd.DataFrame({
    "lag variable": [f"X_t lag {j}" for j in range(p)],
    "weight": weights_ar1
})

print("Theoretical one-step BLP weights for AR(1):")
display(weights_table)
print(f"Theoretical one-step prediction MSE: {pred_mse_ar1:.4f}")
print(f"Innovation variance sigma^2: {sigma2_true:.4f}")

### What should you notice?

The first weight should be close to $\phi=0.75$, and the remaining weights should be essentially zero up to numerical rounding. The prediction MSE should be close to $\sigma^2$, because for an AR(1), the only unpredictable part of $X_{t+1}$ after observing $X_t$ is $W_{t+1}$.

## 4. Empirical one-step forecasting from data

In practice, we do not know $\phi$. We estimate it from data.

For a zero-mean AR(1), the least-squares estimate is

$$
\hat \phi = \frac{\sum_{t=1}^{n-1} x_t x_{t+1}}{\sum_{t=1}^{n-1} x_t^2}.
$$

Then the one-step forecast is

$$
\hat x_{t+1} = \hat \phi x_t.
$$

We will split the data chronologically into a training part and a testing part. This is important: for time series, we should not randomly shuffle observations when evaluating forecasts.

In [ ]:
train_size = 220
x_train = x[:train_size]
x_test = x[train_size:]

# Estimate phi from training data only.
phi_hat = np.sum(x_train[:-1] * x_train[1:]) / np.sum(x_train[:-1] ** 2)
print(f"True phi:      {phi_true:.3f}")
print(f"Estimated phi: {phi_hat:.3f}")

# One-step-ahead forecasts on the test period.
# At each test time, use the actually observed previous value.
actual = x[train_size:]
previous = x[train_size - 1:-1]
forecast_ar1 = phi_hat * previous
forecast_naive = previous  # random walk style baseline: forecast next value by current value
forecast_zero = np.zeros_like(actual)  # mean-zero baseline

results = pd.DataFrame({
    "method": ["estimated AR(1)", "naive previous-value", "zero forecast"],
    "test MSE": [
        mse(actual, forecast_ar1),
        mse(actual, forecast_naive),
        mse(actual, forecast_zero)
    ]
})

display(results)

In [ ]:
plot_t = np.arange(train_size, n)

plt.plot(time, x, label="observed series", alpha=0.55)
plt.axvline(train_size, linestyle="--", label="train/test split")
plt.plot(plot_t, actual, label="test actual", linewidth=2)
plt.plot(plot_t, forecast_ar1, label="one-step AR(1) forecast", linewidth=2)
plt.plot(plot_t, forecast_naive, label="naive previous-value forecast", alpha=0.8)
plt.title("One-step forecasting on the test period")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

### Interpretation checkpoint

The AR(1) forecast is not perfect. Forecasting a noisy process is not about eliminating all error. It is about using dependence in the data to reduce error compared with simpler baselines.

The naive previous-value forecast uses $\hat x_{t+1}=x_t$. This is ideal for a random walk but not necessarily ideal for a stationary AR(1). For $\phi=0.75$, the AR(1) forecast shrinks the current value toward zero.

## 5. Multi-step forecasting and forecast intervals

For a fitted AR(1), the $h$-step forecast from time $T$ is

$$
\hat x_{T+h} = \hat\phi^h x_T.
$$

The forecast uncertainty grows with $h$. If $|\phi|<1$, the $h$-step forecast error variance is approximately

$$
\sigma^2 \left(1 + \phi^2 + \cdots + \phi^{2(h-1)}\right)
= \sigma^2 \frac{1-\phi^{2h}}{1-\phi^2}.
$$

As $h$ increases, the forecast approaches the long-run mean, which is zero in this simulated example.

In [ ]:
# Estimate innovation variance from training residuals.
train_pred = phi_hat * x_train[:-1]
train_resid = x_train[1:] - train_pred
sigma2_hat = np.mean(train_resid**2)

T = train_size - 1
x_T = x[T]
horizon = 30
h = np.arange(1, horizon + 1)

multi_forecast = (phi_hat ** h) * x_T
error_var = sigma2_hat * (1 - phi_hat ** (2 * h)) / (1 - phi_hat**2)
error_sd = np.sqrt(error_var)

upper = multi_forecast + 1.96 * error_sd
lower = multi_forecast - 1.96 * error_sd
future_t = np.arange(T + 1, T + horizon + 1)

plt.plot(time[:train_size], x[:train_size], label="observed through forecast origin")
plt.plot(future_t, multi_forecast, marker="o", label="multi-step forecast")
plt.fill_between(future_t, lower, upper, alpha=0.25, label="approx. 95% forecast interval")
plt.axvline(T, linestyle="--", label="forecast origin")
plt.title("AR(1) multi-step forecast from a fixed origin")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

pd.DataFrame({
    "horizon h": h,
    "forecast": multi_forecast,
    "forecast_sd": error_sd,
    "lower_95": lower,
    "upper_95": upper
}).head(10)

### Interpretation checkpoint

The point forecast moves toward zero because the stationary AR(1) has long-run mean zero. The interval widens because farther-ahead forecasts accumulate more future shocks.

## 6. The orthogonality principle in data

The best linear predictor has a special property: its residuals are uncorrelated with the variables used to make the prediction.

For a lag-based linear regression forecast

$$
\hat x_{t+1} = a_1 x_t + a_2 x_{t-1} + \cdots + a_p x_{t-p+1},
$$

the residual

$$
e_{t+1} = x_{t+1} - \hat x_{t+1}
$$

should have nearly zero sample correlation with each included lag variable, at least on the data used for fitting.

In [ ]:
def make_lag_matrix(series, p):
    """Create lag features [x_t, x_{t-1}, ..., x_{t-p+1}] and target x_{t+1}."""
    series = np.asarray(series)
    X_rows = []
    y_vals = []
    for t in range(p - 1, len(series) - 1):
        X_rows.append([series[t - j] for j in range(p)])
        y_vals.append(series[t + 1])
    return np.asarray(X_rows), np.asarray(y_vals)

p_lags = 5
X_train_lag, y_train_lag = make_lag_matrix(x_train, p_lags)

# Ordinary least squares using numpy.
# No intercept is used because the simulated process has mean zero.
beta_hat = np.linalg.lstsq(X_train_lag, y_train_lag, rcond=None)[0]
y_hat_train = X_train_lag @ beta_hat
resid_train = y_train_lag - y_hat_train

corrs = []
for j in range(p_lags):
    corr = np.corrcoef(resid_train, X_train_lag[:, j])[0, 1]
    corrs.append(corr)

orthogonality_table = pd.DataFrame({
    "lag variable": [f"x_(t-{j})" if j > 0 else "x_t" for j in range(p_lags)],
    "estimated weight": beta_hat,
    "correlation with residual": corrs
})

display(orthogonality_table)

### What should you notice?

The residual correlations should be extremely close to zero on the training data. This is not an accident; it is exactly what least squares does. It is the sample version of the orthogonality principle.

## 7. MA(1) forecasting: why finite-past prediction is different

Now consider the MA(1) process

$$
X_t = W_t + \theta W_{t-1}.
$$

The ACF cuts off after lag 1, but forecasting is subtle because the shock $W_t$ is not directly observed from $X_t$ alone.

For the best linear forecast using $p$ observed lags

$$
\hat X_{t+1} = a_1 X_t + a_2 X_{t-1} + \cdots + a_p X_{t-p+1},
$$

the weights are not simply $[\theta,0,0,\ldots]$. We compute them from the covariance equations.

For MA(1), the autocovariance is

$$
\gamma(0) = \sigma^2(1+\theta^2),
\qquad
\gamma(1) = \sigma^2\theta,
\qquad
\gamma(h)=0 \text{ for } h>1.
$$

In [ ]:
def autocovariance_ma1(theta, sigma2, max_lag):
    gamma = np.zeros(max_lag + 1)
    gamma[0] = sigma2 * (1 + theta**2)
    if max_lag >= 1:
        gamma[1] = sigma2 * theta
    return gamma

theta = 0.7
sigma2 = 1.0
max_p = 8
gamma_ma1 = autocovariance_ma1(theta, sigma2, max_p)

rows = []
for p in range(1, max_p + 1):
    weights, pred_mse = one_step_blp_weights(gamma_ma1, p)
    row = {"p lags used": p, "prediction MSE": pred_mse}
    for j in range(max_p):
        row[f"a_{j+1}"] = weights[j] if j < p else np.nan
    rows.append(row)

ma1_weights = pd.DataFrame(rows)
display(ma1_weights.round(4))

In [ ]:
plt.figure(figsize=(10, 5))
for p in [1, 2, 3, 5, 8]:
    weights, _ = one_step_blp_weights(gamma_ma1, p)
    plt.plot(np.arange(1, p + 1), weights, marker="o", label=f"p={p}")
plt.axhline(0, linewidth=1)
plt.title("Best linear prediction weights for MA(1) using finite past")
plt.xlabel("lag number")
plt.ylabel("weight")
plt.legend()
plt.show()

### Interpretation checkpoint

For an invertible MA(1), older observations can help reconstruct the recent hidden noise. As more lags are allowed, the best linear forecast uses a decaying pattern of weights. This is one reason forecasting MA models is conceptually different from forecasting pure AR models.

## 8. Build a supervised learning table from a time series

Modern machine learning often converts forecasting into supervised learning. A window of past observations becomes a feature vector, and a future observation becomes the target.

For one-step prediction with $p$ lags, each row looks like

$$
(x_t, x_{t-1}, \ldots, x_{t-p+1}) \mapsto x_{t+1}.
$$

This is the bridge between classical best linear prediction and modern sequence models.

In [ ]:
p_lags = 6
X_all, y_all = make_lag_matrix(x, p_lags)

supervised_table = pd.DataFrame(X_all[:8], columns=[f"lag_{j}" for j in range(p_lags)])
supervised_table["target_next"] = y_all[:8]

display(supervised_table)
print("Feature matrix shape:", X_all.shape)
print("Target vector shape:", y_all.shape)

### Student task

Modify `p_lags` and rerun the cell above. Notice how increasing the number of lags changes the feature dimension and reduces the number of available training examples.

## 9. Forecasting with a lag regression model

We now fit a lag regression model with multiple lags and compare it with the AR(1) model.

This is a simple version of a machine-learning forecasting workflow:

1. Choose a forecast target.
2. Build lag features using only past data.
3. Split chronologically.
4. Fit the model on the training period.
5. Evaluate on the future test period.

In [ ]:
p_lags = 6
X_train_ml, y_train_ml = make_lag_matrix(x_train, p_lags)

# Build a test design matrix carefully using the full series, then select rows whose target lies in the test period.
X_full_ml, y_full_ml = make_lag_matrix(x, p_lags)
# In make_lag_matrix, row r corresponds to target time t+1 where t = p_lags-1+r.
target_times = np.arange(p_lags, n)
test_mask = target_times >= train_size
X_test_ml = X_full_ml[test_mask]
y_test_ml = y_full_ml[test_mask]

beta_ml = np.linalg.lstsq(X_train_ml, y_train_ml, rcond=None)[0]
pred_ml = X_test_ml @ beta_ml

comparison = pd.DataFrame({
    "method": ["lag regression with 6 lags", "estimated AR(1)", "naive previous-value", "zero forecast"],
    "test MSE": [
        mse(y_test_ml, pred_ml),
        mse(actual, forecast_ar1),
        mse(actual, forecast_naive),
        mse(actual, forecast_zero)
    ]
})

display(comparison)

coef_table = pd.DataFrame({
    "feature": [f"lag_{j}" for j in range(p_lags)],
    "coefficient": beta_ml
})
display(coef_table)

In [ ]:
plt.plot(np.arange(train_size, n), actual, label="test actual", linewidth=2)
plt.plot(np.arange(train_size, n), forecast_ar1, label="estimated AR(1)", linewidth=2)
plt.plot(target_times[test_mask], pred_ml, label="lag regression with 6 lags", linestyle="--")
plt.title("Comparing one-step forecasts")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

### Interpretation checkpoint

Because the true process is AR(1), the six-lag regression should place most of its weight on the most recent lag. Extra lags may help slightly due to finite-sample randomness, or they may overfit. This is why model comparison and validation matter.

## 10. Independent practice: forecast a more difficult process

Now simulate an AR(2) process:

$$
X_t = 1.2 X_{t-1} - 0.5 X_{t-2} + W_t.
$$

This process has oscillatory behavior because the two lags interact. You will fit lag regressions with different numbers of lags and compare them.

In [ ]:
def simulate_ar2(phi1=1.2, phi2=-0.5, sigma=1.0, n=350, burnin=300, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    total = n + burnin
    w = rng.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(2, total):
        x[t] = phi1 * x[t - 1] + phi2 * x[t - 2] + w[t]
    return x[burnin:]

x2 = simulate_ar2(rng=rng)
train_size2 = 250
x2_train = x2[:train_size2]
x2_test = x2[train_size2:]

plt.plot(x2)
plt.axvline(train_size2, linestyle="--", label="train/test split")
plt.title("Simulated AR(2) process")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

rows = []
for p_lags in [1, 2, 3, 5, 10]:
    Xtr, ytr = make_lag_matrix(x2_train, p_lags)
    Xfull, yfull = make_lag_matrix(x2, p_lags)
    target_times2 = np.arange(p_lags, len(x2))
    mask = target_times2 >= train_size2
    Xte = Xfull[mask]
    yte = yfull[mask]
    beta = np.linalg.lstsq(Xtr, ytr, rcond=None)[0]
    pred = Xte @ beta
    rows.append({
        "number of lags": p_lags,
        "test MSE": mse(yte, pred),
        "first coefficient": beta[0],
        "second coefficient": beta[1] if p_lags >= 2 else np.nan
    })

ar2_comparison = pd.DataFrame(rows)
display(ar2_comparison)

### Reflection question

Which number of lags worked best? Did the model with two lags perform well? Why should that happen for an AR(2) process?

## 11. AI-assisted study prompts

Use an AI assistant only after you have tried the lab yourself. Good prompts ask the assistant to explain, diagnose, or check your reasoning rather than simply give an answer.

### Prompt 1: Concept check

> I am studying best linear prediction for time series. Explain why the forecast error should be orthogonal to the variables used in the forecast. Use the one-step-ahead linear forecast $a_1X_t+a_2X_{t-1}$ as the example.

### Prompt 2: Code review

> Here is my lag-regression forecasting code. Check whether I accidentally used future information in the features or train/test split. Explain each possible leakage issue.

### Prompt 3: Interpretation

> I simulated an AR(1) process with positive phi. My multi-step forecasts move toward zero. Explain why this happens mathematically and visually.

### Prompt 4: Compare AR and MA forecasting

> Explain why one-step forecasting for an MA(1) process is different from one-step forecasting for an AR(1) process, even though both are linear time series models.

## 12. Exercises

### Exercise 1

Change the AR(1) parameter to $\phi=0.3$ and then to $\phi=0.95$. For each value:

1. Simulate a new series.
2. Estimate $\phi$ from training data.
3. Compare the AR(1), naive, and zero forecasts.
4. Explain how persistence changes forecast difficulty.

### Exercise 2

For the AR(1) process, compute the theoretical BLP weights using $p=1,2,3,10$ lags. Explain why the extra lags receive weights close to zero.

### Exercise 3

For the MA(1) process, try $\theta=0.3$, $\theta=0.7$, and $\theta=-0.7$. Plot the BLP weights. How does the sign and magnitude of $\theta$ affect the weights?

### Exercise 4

Simulate an AR(2) process with your own coefficients. Compare lag regressions with 1, 2, 5, and 10 lags. Which one performs best?

### Exercise 5

Write a short paragraph explaining the difference between:

- the ideal conditional expectation forecast,
- the best linear predictor,
- a fitted lag regression model,
- and a naive forecast.

## 13. Lab summary

In this lab, you learned that forecasting is not just curve fitting. It depends on the information available at the forecast origin and on the dependence structure of the process.

Key takeaways:

1. The conditional expectation is the best mean-square forecast in theory.
2. The best linear predictor is obtained by solving covariance-based normal equations.
3. The forecast error is orthogonal to the variables used in the best linear forecast.
4. AR models often have simple recursive forecasts.
5. MA models can require reconstructing hidden shocks from observed history.
6. Lag-based supervised learning is a practical bridge from classical time series to modern machine learning.
7. Forecast evaluation must respect chronological order.